# Week 7 — distortion sweep visualization

Loads the manifest, picks 4 sample tiles, and renders one grid per distortion plus a per-distortion SNR histogram.

Pre-requisite: run `python -m scripts.apply_distortions --clean-root data/clean --out-root data/distorted --manifest results/distortion_manifest.csv` from the repo root before opening this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
from src.figures import plot_distortion_grid

REPO = Path.cwd().parent  # notebooks/ → repo root
CLEAN  = REPO / "data" / "clean"
DIST   = REPO / "data" / "distorted"
OUTFIG = REPO / "outputs" / "figures"
OUTFIG.mkdir(parents=True, exist_ok=True)
MANIFEST = REPO / "results" / "distortion_manifest.csv"

df = pd.read_csv(MANIFEST)
samples = sorted(df["name"].unique())[:4]
samples

In [ ]:
plot_distortion_grid(
    clean_root=CLEAN, distorted_root=DIST,
    distortion="haze", levels=["0.5", "1.5", "3.0"],
    sample_names=samples,
    out_path=OUTFIG / "distorted_grid_haze.png",
)

In [ ]:
plot_distortion_grid(
    clean_root=CLEAN, distorted_root=DIST,
    distortion="jpeg", levels=["20", "5", "1"],
    sample_names=samples,
    out_path=OUTFIG / "distorted_grid_jpeg.png",
)

In [ ]:
plot_distortion_grid(
    clean_root=CLEAN, distorted_root=DIST,
    distortion="noise", levels=["5", "25", "50"],
    sample_names=samples,
    out_path=OUTFIG / "distorted_grid_noise.png",
)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (d, sub) in zip(axes, df.groupby("distortion")):
    finite = sub[sub["snr_db"].apply(lambda x: x < 1e9)]
    ax.hist(finite["snr_db"], bins=20, color="steelblue", edgecolor="white")
    ax.set_title(f"{d}: SNR (dB) across sweep")
    ax.set_xlabel("SNR (dB)")
plt.tight_layout()
plt.savefig(OUTFIG / "distorted_snr_hist.png", dpi=120)
plt.show()